# 00 - Project setup and data inventory

This notebook establishes the project paths and creates a reproducible inventory of the supplied CSV files. It checks file presence, row/column counts, headers, inferred column types, missing values, date ranges, and likely key uniqueness. It does **not** clean or overwrite source data.

## Run instructions
Run every cell from top to bottom. The project root is found by searching the current working directory and its parents for `data/student_data`. Inventory outputs are written to `outputs/data_inventory/`.

## Working assumptions from the problem statement
- `Date` is the authoritative period; `Year` and `Month` are consistency checks only.
- A repeated `Record_ID` is a duplicate load; the latest `Load_Timestamp` is authoritative when cleaning in a later notebook.
- A missing row is not equivalent to zero sales.
- The raw train may contain multiple records for a period/SKU/region; measure and investigate duplicates before aggregating or removing anything.
- The expected row counts below are checks against the supplied brief, not permission to silently drop or invent records.

## Expected files
| File | Expected data rows | Purpose |
|---|---:|---|
| `train.csv` | 248,248 | Historical sales, Jan 2019-Oct 2026 |
| `test.csv` | 58,219 | Forecast keys, Nov 2026-May 2028 |
| `macro_indicators.csv` | 114 | Monthly economic indicators |
| `sku_master.csv` | 601 | SKU attributes and lifecycle |
| `events_calendar.csv` | 68 | Known business events |
| `inventory_stockouts.csv` | 5,830 | Partial record of constrained supply |
| `sample_submission.csv` | 5 | Required submission example |

This notebook uses Python's standard library only, so it can run before analysis packages are installed.

In [ ]:
from collections import defaultdict
from datetime import date, datetime
import csv
import json
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data" / "student_data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/student_data from the notebook working directory. "
        "Open/run this notebook from the project folder or one of its subfolders."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "student_data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_inventory"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {
    "train.csv": (248_248, [
        "Record_ID", "Date", "Year", "Month", "Product_Family", "Business_Unit",
        "SKU_ID", "SKU_Description", "Customer_Name", "Sales_Channel", "Region",
        "Units_Sold", "Unit_Price_USD", "Revenue_USD", "Load_Timestamp",
    ]),
    "test.csv": (58_219, [
        "Date", "Year", "Month", "Product_Family", "Business_Unit", "SKU_ID",
        "SKU_Description", "Customer_Name", "Sales_Channel", "Region",
    ]),
    "macro_indicators.csv": (114, [
        "Date", "Year", "Month", "ISM_Manufacturing_PMI", "Auto_Production_YoY_Pct",
        "Vehicle_Inventory_Days", "Raw_Material_Cost_Index", "New_Vehicle_Orders_Index",
        "Industrial_Util_Rate_Pct", "Fuel_Price_Index", "EV_Adoption_Index",
    ]),
    "sku_master.csv": (601, [
        "SKU_ID", "Product_Family", "Business_Unit", "SKU_Description", "Size_Class",
        "Powertrain_Application", "Launch_Date", "EOL_Date", "Replaces_SKU",
        "Lifecycle_Status",
    ]),
    "events_calendar.csv": (68, [
        "Event_ID", "Start_Date", "End_Date", "Scope_Type", "Scope_Value",
        "Event_Type", "Description",
    ]),
    "inventory_stockouts.csv": (5_830, [
        "Date", "SKU_ID", "Region", "Requested_Units", "Carried_In_Units",
        "Shipped_Units", "Backordered_Units", "Lost_Units", "Fill_Rate_Pct",
        "Stockout_Days", "Reason",
    ]),
    "sample_submission.csv": (5, [
        "Date", "SKU_ID", "Region", "Units_Sold_Forecast",
    ]),
}

DATE_COLUMNS = {
    "Date", "Start_Date", "End_Date", "Launch_Date", "EOL_Date", "Load_Timestamp",
}
KEYS_TO_CHECK = {
    "train.csv": [("Record_ID",), ("Date", "SKU_ID", "Region")],
    "test.csv": [("Date", "SKU_ID", "Region")],
    "macro_indicators.csv": [("Date",)],
    "sku_master.csv": [("SKU_ID",)],
    "events_calendar.csv": [("Event_ID",)],
    "inventory_stockouts.csv": [("Date", "SKU_ID", "Region")],
    "sample_submission.csv": [("Date", "SKU_ID", "Region")],
}
EXPECTED_DATE_RANGES = {
    "train.csv": (date(2019, 1, 1), date(2026, 10, 1)),
    "test.csv": (date(2026, 11, 1), date(2028, 5, 1)),
    "macro_indicators.csv": (date(2019, 1, 1), date(2028, 5, 1)),
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Input folder: {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Input folder does not exist: {DATA_DIR}")

## Profile the files

The profiler streams each CSV rather than loading the full dataset into memory. `inferred_type` is a lightweight audit inference (date, integer, number, text, mixed/invalid date, or empty); it does not replace deliberate type conversion in later notebooks. It accepts common alternate date spellings for profiling, but counts noncanonical values so they can be standardized later. Blank/whitespace-only cells count as missing.

In [ ]:
def parse_date_value(value):
    value = (value or "").strip()
    if not value:
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00")).date()
    except ValueError:
        pass

    for date_format in (
        "%Y-%m", "%Y/%m", "%b %Y", "%B %Y", "%b-%Y", "%B-%Y",
        "%m-%Y", "%m/%Y", "%m/%d/%Y", "%d/%m/%Y", "%d-%m-%Y",
    ):
        try:
            return datetime.strptime(value, date_format).date()
        except ValueError:
            continue
    try:
        return date.fromisoformat(value)
    except ValueError:
        return None


def is_canonical_date_value(value, parsed):
    return (
        value == parsed.isoformat()
        or value == parsed.strftime("%Y-%m")
        or value.startswith(parsed.isoformat() + "T")
        or value.startswith(parsed.isoformat() + " ")
    )


def is_number(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False


def profile_csv(file_name, expected_rows, expected_header):
    path = DATA_DIR / file_name
    if not path.is_file():
        return None, [], [f"Missing required file: {file_name}"]

    issues = []
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        header = reader.fieldnames or []
        if len(header) != len(set(header)):
            issues.append(f"{file_name}: duplicate column names in header")
        if header != expected_header:
            issues.append(f"{file_name}: header differs from the expected schema")

        column_state = {
            name: {
                "nonempty": 0,
                "missing": 0,
                "integer_like": 0,
                "number_like": 0,
                "date_valid": 0,
                "date_invalid": 0,
                "date_noncanonical": 0,
                "samples": [],
                "date_min": None,
                "date_max": None,
            }
            for name in header
        }
        row_count = 0
        malformed_rows = 0
        key_counts = {
            key_fields: defaultdict(int)
            for key_fields in KEYS_TO_CHECK.get(file_name, [])
            if all(field in header for field in key_fields)
        }
        period_state = {
            "date_min": None,
            "date_max": None,
            "out_of_expected_range_rows": 0,
            "year_month_mismatch_rows": 0,
            "year_month_mismatch_examples": [],
        }

        for row in reader:
            row_count += 1
            if None in row:
                malformed_rows += 1
            parsed_period_date = None
            for name in header:
                value = (row.get(name) or "").strip()
                state = column_state[name]
                if not value:
                    state["missing"] += 1
                    continue

                state["nonempty"] += 1
                if len(state["samples"]) < 3 and value not in state["samples"]:
                    state["samples"].append(value)
                if is_number(value):
                    state["number_like"] += 1
                    if value.lstrip("+-").isdigit():
                        state["integer_like"] += 1

                if name in DATE_COLUMNS:
                    parsed = parse_date_value(value)
                    if name == "Date":
                        parsed_period_date = parsed
                    if parsed is None:
                        state["date_invalid"] += 1
                    else:
                        state["date_valid"] += 1
                        if not is_canonical_date_value(value, parsed):
                            state["date_noncanonical"] += 1
                        if state["date_min"] is None or parsed < state["date_min"]:
                            state["date_min"] = parsed
                        if state["date_max"] is None or parsed > state["date_max"]:
                            state["date_max"] = parsed

            if parsed_period_date is not None:
                period_month = date(parsed_period_date.year, parsed_period_date.month, 1)
                if period_state["date_min"] is None or period_month < period_state["date_min"]:
                    period_state["date_min"] = period_month
                if period_state["date_max"] is None or period_month > period_state["date_max"]:
                    period_state["date_max"] = period_month
                expected_range = EXPECTED_DATE_RANGES.get(file_name)
                if expected_range and not expected_range[0] <= period_month <= expected_range[1]:
                    period_state["out_of_expected_range_rows"] += 1
                if "Year" in row and "Month" in row:
                    try:
                        matches_year_month = (
                            parsed_period_date.year == int(row["Year"])
                            and parsed_period_date.month == int(row["Month"])
                        )
                    except (TypeError, ValueError):
                        matches_year_month = False
                    if not matches_year_month:
                        period_state["year_month_mismatch_rows"] += 1
                        if len(period_state["year_month_mismatch_examples"]) < 5:
                            period_state["year_month_mismatch_examples"].append({
                                "Date": row.get("Date", ""),
                                "Year": row.get("Year", ""),
                                "Month": row.get("Month", ""),
                                "Record_ID": row.get("Record_ID", ""),
                            })

            for key_fields, counts in key_counts.items():
                key = tuple((row.get(field) or "").strip() for field in key_fields)
                if all(key):
                    counts[key] += 1

    if row_count != expected_rows:
        issues.append(
            f"{file_name}: found {row_count:,} data rows; brief expects {expected_rows:,}"
        )
    if malformed_rows:
        issues.append(f"{file_name}: {malformed_rows:,} rows have extra fields")
    if period_state["out_of_expected_range_rows"]:
        issues.append(
            f"{file_name}.Date: {period_state['out_of_expected_range_rows']:,} rows outside the expected month window"
        )
    if period_state["year_month_mismatch_rows"]:
        issues.append(
            f"{file_name}: {period_state['year_month_mismatch_rows']:,} Date vs Year/Month mismatches"
        )

    inventory_row = {
        "file": file_name,
        "rows": row_count,
        "columns": len(header),
        "size_bytes": path.stat().st_size,
        "expected_rows": expected_rows,
        "row_count_matches_brief": row_count == expected_rows,
        "malformed_rows_extra_fields": malformed_rows,
        "date_min": period_state["date_min"].isoformat() if period_state["date_min"] else "",
        "date_max": period_state["date_max"].isoformat() if period_state["date_max"] else "",
        "expected_date_min": EXPECTED_DATE_RANGES[file_name][0].isoformat()
        if file_name in EXPECTED_DATE_RANGES else "",
        "expected_date_max": EXPECTED_DATE_RANGES[file_name][1].isoformat()
        if file_name in EXPECTED_DATE_RANGES else "",
        "out_of_expected_range_rows": period_state["out_of_expected_range_rows"],
        "year_month_mismatch_rows": period_state["year_month_mismatch_rows"],
        "year_month_mismatch_examples": json.dumps(
            period_state["year_month_mismatch_examples"]
        ),
    }
    column_rows = []
    for name, state in column_state.items():
        nonempty = state["nonempty"]
        if name in DATE_COLUMNS:
            inferred_type = "date" if not state["date_invalid"] else "date/mixed"
            if not nonempty:
                inferred_type = "empty"
            elif not state["date_valid"]:
                inferred_type = "text/invalid-date"
        elif not nonempty:
            inferred_type = "empty"
        elif state["number_like"] == nonempty:
            inferred_type = (
                "integer" if state["integer_like"] == nonempty else "number"
            )
        elif state["number_like"]:
            inferred_type = "mixed-number/text"
        else:
            inferred_type = "text"

        if name in DATE_COLUMNS and state["date_invalid"]:
            issues.append(
                f"{file_name}.{name}: {state['date_invalid']:,} nonblank values are not parseable dates"
            )
        column_rows.append({
            "file": file_name,
            "column": name,
            "inferred_type": inferred_type,
            "nonempty_values": nonempty,
            "missing_values": state["missing"],
            "missing_pct": round(100 * state["missing"] / row_count, 4) if row_count else 0,
            "invalid_date_values": state["date_invalid"],
            "noncanonical_date_values": state["date_noncanonical"],
            "date_min": state["date_min"].isoformat() if state["date_min"] else "",
            "date_max": state["date_max"].isoformat() if state["date_max"] else "",
            "sample_values": " | ".join(state["samples"]),
        })

    key_rows = []
    for key_fields, counts in key_counts.items():
        duplicate_groups = sum(count > 1 for count in counts.values())
        duplicate_extra_rows = sum(count - 1 for count in counts.values() if count > 1)
        key_rows.append({
            "file": file_name,
            "key": "+".join(key_fields),
            "distinct_nonmissing_keys": len(counts),
            "duplicate_key_groups": duplicate_groups,
            "duplicate_extra_rows": duplicate_extra_rows,
            "unique": duplicate_extra_rows == 0 and len(counts) == row_count,
        })
    return inventory_row, column_rows, issues, key_rows


## Run checks and save the inventory

Duplicate counts are diagnostic only. In particular, do not drop duplicate date/SKU/region rows here: the next cleaning workstream must first determine whether they are repeated loads, legitimate source records, or aggregation issues.

In [ ]:
inventory_rows = []
column_rows = []
key_rows = []
schema_issues = []

for file_name, (expected_rows, expected_header) in EXPECTED_FILES.items():
    result = profile_csv(file_name, expected_rows, expected_header)
    if result[0] is None:
        schema_issues.extend(result[2])
        continue
    inventory_row, file_columns, file_issues, file_keys = result
    inventory_rows.append(inventory_row)
    column_rows.extend(file_columns)
    schema_issues.extend(file_issues)
    key_rows.extend(file_keys)

expected_names = set(EXPECTED_FILES)
present_csv_names = {path.name for path in DATA_DIR.glob("*.csv")}
unexpected_csvs = sorted(present_csv_names - expected_names)
if unexpected_csvs:
    schema_issues.append(f"Unexpected CSV files found: {', '.join(unexpected_csvs)}")

def write_csv(path, rows):
    if not rows:
        return
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


write_csv(OUTPUT_DIR / "file_inventory.csv", inventory_rows)
write_csv(OUTPUT_DIR / "column_profiles.csv", column_rows)
write_csv(OUTPUT_DIR / "key_uniqueness.csv", key_rows)

audit = {
    "project_root": str(PROJECT_ROOT),
    "data_directory": str(DATA_DIR),
    "expected_files": sorted(expected_names),
    "present_csv_files": sorted(present_csv_names),
    "unexpected_csv_files": unexpected_csvs,
    "file_inventory": inventory_rows,
    "key_uniqueness": key_rows,
    "schema_issues": schema_issues,
    "assumptions": [
        "Date is authoritative; Year and Month are consistency checks.",
        "For repeated Record_ID values, latest Load_Timestamp wins during cleaning.",
        "Missing rows are not zero sales.",
        "Do not remove duplicate period/SKU/region rows until their cause is investigated.",
        "This notebook profiles source data and does not modify it.",
    ],
}
with (OUTPUT_DIR / "inventory_audit.json").open("w", encoding="utf-8") as handle:
    json.dump(audit, handle, indent=2)

print("FILE INVENTORY")
for row in inventory_rows:
    status = "OK" if row["row_count_matches_brief"] else "CHECK ROW COUNT"
    print(
        f"{row['file']:<28} {row['rows']:>9,} rows  "
        f"{row['columns']:>2} columns  {row['size_bytes']:>10,} bytes  {status}"
    )

print("\nKEY UNIQUENESS CHECKS")
for row in key_rows:
    print(
        f"{row['file']:<28} {row['key']:<30} "
        f"duplicate groups={row['duplicate_key_groups']:,}, "
        f"extra rows={row['duplicate_extra_rows']:,}, unique={row['unique']}"
    )

print("\nSCHEMA / PERIOD ISSUES")
if schema_issues:
    for issue in schema_issues:
        print(f"- {issue}")
else:
    print("No missing files, header mismatches, row-count mismatches, or invalid date values detected.")

print(f"\nSaved inventory reports to: {OUTPUT_DIR}")

## How to interpret the reports
- `file_inventory.csv`: one row per expected source file, including dimensions, byte size, and comparison with the brief.
- `column_profiles.csv`: inferred type, blank count/percentage, date range where applicable, and example values for each column.
- `key_uniqueness.csv`: duplicate key groups and extra rows for each proposed natural key. These results identify work for the cleaning notebook; they are not automatic delete instructions.
- `inventory_audit.json`: machine-readable summary, assumptions, and schema issues.

Before continuing, inspect the profiles and key checks. If row counts or headers differ from the brief, investigate the source version rather than changing the expectations to make the check pass.